# Linear Regression

This notebook accompanies the **ML Viz** lesson on linear regression.
We'll implement OLS, Ridge, and Lasso from scratch.

**Companion lesson:** https://ml-viz.vercel.app/courses/linear-regression/01-linear-regression

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## OLS: Closed-Form Solution

$$w^* = (X^T X)^{-1} X^T y$$

In [ ]:
np.random.seed(42)
n = 100
X = 2 * np.random.randn(n, 1) + 1
y = 3.5 * X.squeeze() + 1.2 + 0.5 * np.random.randn(n)

X_b = np.c_[np.ones(n), X]  # add bias column -> shape (n, 2)

# --- Closed-form normal equation: w = (X^T X)^-1 X^T y ---
w_ols = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print("Closed-form:    b = {:.3f}, w = {:.3f}".format(w_ols[0], w_ols[1]))

# --- Gradient descent on the SAME MSE loss ---
# grad of (1/n)||Xw - y||^2  =  (2/n) X^T (Xw - y)
def gradient_descent(X, y, lr=0.05, n_iter=2000):
    w = np.zeros(X.shape[1])
    m = X.shape[0]
    for _ in range(n_iter):
        grad = (2.0 / m) * X.T @ (X @ w - y)
        w = w - lr * grad
    return w

w_gd = gradient_descent(X_b, y)
print("Gradient desc.: b = {:.3f}, w = {:.3f}".format(w_gd[0], w_gd[1]))

# --- Residuals and R^2 ---
y_hat = X_b @ w_ols
resid = y - y_hat
ss_res = np.sum(resid ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print("R^2 = {:.4f}".format(r2))

# --- Plot: fit (left) and residuals (right) ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X, y, c='#818cf8', s=15, alpha=0.6, label='Data')
x_line = np.linspace(X.min(), X.max(), 100)
ax1.plot(x_line, w_ols[0] + w_ols[1] * x_line,
         color='#14b8a6', linewidth=2, label='OLS fit')
ax1.legend()
ax1.set_title('Ordinary Least Squares (R^2 = {:.3f})'.format(r2), color='white')
ax1.set_xlabel('x'); ax1.set_ylabel('y')

ax2.scatter(y_hat, resid, c='#f59e0b', s=15, alpha=0.6)
ax2.axhline(0, color='#94a3b8', linewidth=1, linestyle='--')
ax2.set_title('Residuals vs. fitted', color='white')
ax2.set_xlabel('predicted y'); ax2.set_ylabel('residual (y - y_hat)')

plt.tight_layout()
plt.show()

## Ridge vs Lasso

Ridge (L2): $\mathcal{L} = \|y - Xw\|^2 + \lambda\|w\|^2$
Lasso (L1): $\mathcal{L} = \|y - Xw\|^2 + \lambda\|w\|_1$

In [ ]:
# Ridge closed form: w = (X^T X + lam*I)^-1 X^T y
def ridge(X, y, lam):
    p = X.shape[1]
    return np.linalg.inv(X.T @ X + lam * np.eye(p)) @ X.T @ y

# Lasso via coordinate descent with soft-thresholding
def soft_threshold(rho, lam):
    if rho > lam:
        return rho - lam
    if rho < -lam:
        return rho + lam
    return 0.0

def lasso_coordinate_descent(X, y, lam, n_iter=500):
    m, p = X.shape
    w = np.zeros(p)
    for _ in range(n_iter):
        for j in range(p):
            # residual excluding feature j's current contribution
            r_j = y - X @ w + X[:, j] * w[j]
            rho = X[:, j] @ r_j / m
            z_j = X[:, j] @ X[:, j] / m
            w[j] = soft_threshold(rho, lam) / z_j
    return w

# Show Ridge shrinking the fit as lambda grows
lambdas = [0.01, 0.1, 1.0, 5.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(4 * len(lambdas), 4), sharey=True)
fig.suptitle('Ridge Regression: Effect of lambda', color='white', fontsize=13, y=1.02)

for ax, lam in zip(axes, lambdas):
    w_r = ridge(X_b, y, lam)
    ax.scatter(X, y, c='#818cf8', s=10, alpha=0.4)
    ax.plot(x_line, w_r[0] + w_r[1] * x_line, color='#14b8a6', linewidth=2)
    ax.set_title('lambda = {}'.format(lam), color='white', fontsize=11)
plt.tight_layout()
plt.show()

## Worked example: OLS by hand

From the lesson: $X = [[1,1],[1,2],[1,3]]$, $y = [2,3,5]$. Solve $w^* = (X^\top X)^{-1} X^\top y$.

In [ ]:
X = np.array([[1, 1], [1, 2], [1, 3]])
y = np.array([2, 3, 5])

# Normal equation, by hand the answer is [0.333, 1.5]
w = np.linalg.inv(X.T @ X) @ X.T @ y
print("w* =", w.round(3), "  (intercept, slope)")

# Residuals and R^2 (matches the lesson's hand computation)
y_hat = X @ w
resid = y - y_hat
ss_res = np.sum(resid ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
print("residuals =", resid.round(3))
print("SS_res = {:.3f}, SS_tot = {:.3f}".format(ss_res, ss_tot))
print("R^2 = {:.3f}".format(1 - ss_res / ss_tot))
print("prediction at x=4:", round(np.array([1, 4]) @ w, 3))

## Regularization paths: Ridge vs Lasso

As the penalty $\alpha$ grows, Ridge shrinks weights smoothly toward zero; Lasso drives some to **exactly** zero (feature selection).

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.datasets import make_regression

Xr, yr = make_regression(n_samples=100, n_features=8, n_informative=3,
                         noise=10, random_state=0)
alphas = np.logspace(-2, 2, 30)
ridge = np.array([Ridge(a).fit(Xr, yr).coef_ for a in alphas])
lasso = np.array([Lasso(a).fit(Xr, yr).coef_ for a in alphas])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(alphas, ridge); ax[0].set_xscale('log'); ax[0].set_title('Ridge (L2)')
ax[1].plot(alphas, lasso); ax[1].set_xscale('log'); ax[1].set_title('Lasso (L1)')
for a in ax: a.set_xlabel('alpha'); a.set_ylabel('coefficient')
plt.tight_layout(); plt.show()

## Key takeaways

- Linear regression fits $y = Xw$ by minimizing squared error.
- **OLS** has a closed form: $w^* = (X^\top X)^{-1} X^\top y$.
- **Ridge (L2)** shrinks weights smoothly; **Lasso (L1)** zeros some out (sparse models).
- Regularization trades a little bias for much lower variance — key against overfitting.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — OLS via the normal equations

The least-squares weights solve the **normal equations**:

$$X^\top X \, \mathbf{w} = X^\top \mathbf{y}$$

Implement the fit with `np.linalg.solve` (never invert explicitly — solving is faster and numerically safer). The checks recover a perfect line and verify the property the whole method is named for: the residuals are **orthogonal to every column** of $X$.

In [ ]:
def ols_fit(X, y):
    """Least-squares weights via the normal equations."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): solve (X^T X) w = X^T y  (hint: np.linalg.solve)
    return ...

In [ ]:
# Checks — run me
x = np.array([0.0, 1.0, 2.0, 3.0])
X = np.column_stack([np.ones_like(x), x])
y = 2 * x + 1
assert np.allclose(ols_fit(X, y), [1, 2]), "perfect line y = 2x + 1 -> intercept 1, slope 2"

rng = np.random.default_rng(0)
Xr = np.column_stack([np.ones(50), rng.standard_normal((50, 2))])
yr = Xr @ np.array([0.5, -1.0, 2.0]) + 0.1 * rng.standard_normal(50)
wr = ols_fit(Xr, yr)
assert np.allclose(Xr.T @ (yr - Xr @ wr), 0, atol=1e-9), "residuals must be orthogonal to every column of X"
assert np.allclose(wr, np.linalg.lstsq(Xr, yr, rcond=None)[0]), "must match np.linalg.lstsq"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ols_fit(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    return np.linalg.solve(X.T @ X, X.T @ y)
```

</details>

### Exercise 2 — Ridge regression in closed form

Adding the $L_2$ penalty $\lambda \lVert \mathbf{w} \rVert^2$ only changes the normal equations by a diagonal nudge:

$$(X^\top X + \lambda I) \, \mathbf{w} = X^\top \mathbf{y}$$

The checks confirm the three behaviors from the lesson: $\lambda = 0$ recovers OLS, moderate $\lambda$ **shrinks** the weights, and $\lambda \to \infty$ crushes them to zero.

In [ ]:
def ridge_fit(X, y, lam):
    """Ridge weights: solve (X^T X + lam * I) w = X^T y."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    d = X.shape[1]

    # TODO(you): the regularized normal equations (hint: np.eye(d))
    return ...

In [ ]:
# Checks — run me
assert np.allclose(ridge_fit(Xr, yr, 0.0), wr), "lambda = 0 recovers OLS"

w10 = ridge_fit(Xr, yr, 10.0)
assert np.linalg.norm(w10) < np.linalg.norm(wr), "the penalty shrinks the weights"

w_huge = ridge_fit(Xr, yr, 1e8)
assert np.linalg.norm(w_huge) < 1e-3, "lambda -> infinity crushes the weights to ~0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def ridge_fit(X, y, lam):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    d = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(d), X.T @ y)
```

</details>